# OPM (Order Point Management) Analysis Generator

Builds a three-tab `OPM_Analysis_Output.xlsx` workbook from:

- **`pmovdcE_FG_3Aug26.xlsx`** — parts movement export → Brc, Agc, P/N, OH, OO, DN Price, Last Sales, Last GRR, Total Calls (sum of C-1:C-12)
- **`FD_FG_3Aug26.xlsx`** (sheet `Branch`) → `FD_final` per Brc + P/N

Output tabs:

- **`Parameters`** — the OPM input parameter grid laid out exactly like your original input sheet:
  four quadrants (A/B/C/D), each with its own Calls range and 5-row Min Price / Max Price →
  Service Level table, plus an agency-level OCLT cell and a per-branch LT table. The PF table is
  hardcoded directly into the formula since it never changes, so it isn't duplicated here.
- **`Data`** — one row per Brc + P/N with live formulas for **OPM, SL, PF, ExDlt, OC, Min, ROP, Max**
  (ExDlt through Max are whole numbers) that recalculate automatically whenever you edit the yellow
  cells on the Parameters tab. **Branch 20 is replaced by a `National` row per P/N**: OH, OO, and
  Total Calls are summed across every branch, Last Sales/Last GRR are taken from branch 20's own
  data, and FD_final is pulled from the FD workbook's separate `National` sheet instead of the
  per-branch `Branch` sheet.
- **`Dashboard`** — a single-part lookup card: pick Agency/Branch/PN (Branch now includes
  `National`) and see OH, OO, DN Price,
  Total Calls, FD Final, Min, Max, ExDlt, ROP, Last Sales, Last GRR, plus four checkbox indicators
  (Overstock / Deadstock / Critical / Reorder).

Formula logic:

| Field | Formula |
|---|---|
| OPM | Which quadrant's (A/B/C/D) Calls range the part's Total Calls falls into |
| SL | Service Level looked up from DN Price bracket within that quadrant |
| PF | Price Factor looked up from SL (hardcoded table) |
| ExDlt | `ROUND(FD_final * LT / 30, 0)` |
| OC | `ROUND(OCLT * FD_final / 14, 0)` |
| Min | `ROUND(PF * SQRT(ExDlt) + OC, 0)` |
| ROP | `ROUND(Min + ExDlt, 0)` |
| Max | `ROUND(Min + ROP, 0)` |

Dashboard checkbox rules:

| Condition | Rule |
|---|---|
| Overstock | `OH > Max` |
| Deadstock | Last Sales or Last GRR older than 3 years for National, 2.5 years for every other branch |
| Critical | `OH < ExDlt` |
| Reorder | `OH < ROP` |

Re-run this notebook any time you have a new `pmovdcE` / `FD` export — it always regenerates the
Data sheet fresh from the source files. LT/OCLT you've entered previously are **not** carried over
automatically since they live only in the generated Excel file; use the "carry forward" option in
cell 2 to reuse them, or keep a copy of your filled-in Parameters tab.

## 1. Setup

In [1]:
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

# ---- file paths -------------------------------------------------
PMOVDCE_FILE = "pmovdcE FG 3Aug26 Edited.xlsx"   # source: parts movement export
FD_FILE      = "FD FG 3Aug26.xlsx"        # source: forecast demand workbook (sheet 'Branch')
OUTPUT_FILE  = "Preliminary Dashboard 1.1 agc 23.xlsx"

# If you previously filled in LT/OCLT on an earlier run and want to carry those values forward,
# point this at that file. Leave as None to start with blank LT/OCLT inputs.
PRIOR_PARAMETERS_FILE = None   # e.g. "OPM_Analysis_Output.xlsx"


## 2. Load & merge source data

Branch 20 is replaced by a **National** rollup: OH, OO, and Total Calls are summed across every
branch; Last Sales and Last GRR are taken from branch 20's own data (not aggregated); and FD_final
comes from the FD workbook's separate `National` sheet (not the per-branch `Branch` sheet).

In [2]:
# --- pmovdcE: Brc, Agc, P/N, OH, OO, DN Price, Last Sales, Last GRR, Total Calls ---
pm = pd.read_excel(PMOVDCE_FILE, sheet_name=0, header=4)
pm.columns = [str(c).strip() for c in pm.columns]

call_cols = [f"C-{i}" for i in range(1, 13)]
pm["Total Calls"] = pm[call_cols].sum(axis=1)
pm["P/N"] = pm["P/N"].astype(str).str.strip()
pm["Brc"] = pm["Brc"].astype(int)

demand_cols = [f"D-{i}" for i in range(1, 13)]
pm["Total Demands"] = pm[demand_cols].sum(axis=1)

data = pm[["Brc", "Agc", "P/N", "OH", "OO", "DN Price", "Last Sales", "Last GRR", "Total Calls", "Total Demands"]].copy()
data.columns = ["Brc", "Agc", "PN", "OH", "OO", "DN_Price", "Last_Sales", "Last_GRR", "Total_Calls", "Total_Demands"]

# --- FD_final per Brc + P/N, from the FD workbook's 'Branch' sheet (real branches only) ---
fd_branch = pd.read_excel(FD_FILE, sheet_name="Branch")
fd_branch.columns = [str(c).strip() for c in fd_branch.columns]
fd_branch = fd_branch[fd_branch["brc"] != "National"].copy()
fd_branch["brc"] = fd_branch["brc"].astype(int)
fd_branch["p/n"] = fd_branch["p/n"].astype(str).str.strip()
fd_branch_small = fd_branch[["brc", "p/n", "FD_final"]].rename(columns={"brc": "Brc", "p/n": "PN"})

# --- National FD_final, from the FD workbook's 'National' sheet ---
fd_nat = pd.read_excel(FD_FILE, sheet_name="National")
fd_nat.columns = [str(c).strip() for c in fd_nat.columns]
fd_nat["p/n"] = fd_nat["p/n"].astype(str).str.strip()
fd_nat_small = fd_nat[["p/n", "FD_final"]].rename(columns={"p/n": "PN", "FD_final": "FD_final_National"})

# =================================================================
# Branch 20 -> National rollup:
#   OH, OO, DN Price, Last Sales, Last GRR -> Branch 20
#   FD_final                               -> National sheet
#   Total Calls                            -> National (sum of C-1:C-12)
#   Total Demands                          -> National (sum of D-1:D-12)
# =================================================================
# ---------- branch 20 values ----------
branch20 = data[data["Brc"] == 20].copy()

# ---------- national totals ----------
national_calls = (
    data.groupby("PN", as_index=False)["Total_Calls"]
    .sum()
)

national_demands = (
    data.groupby("PN", as_index=False)["Total_Demands"]
    .sum()
)

# ---------- build National row ----------
national = branch20[["PN","Agc","OH","OO","DN_Price","Last_Sales","Last_GRR",]
].copy()

national["Brc"] = "National"

national = national.merge(
    national_calls,
    on="PN",
    how="left"
)

national = national.merge(
    national_demands,
    on="PN",
    how="left"
)

national = national.merge(
    fd_nat_small,
    on="PN",
    how="left"
)

national["FD_final"] = national["FD_final_National"].fillna(0)
national.drop(columns="FD_final_National", inplace=True)

national = national[["Brc","Agc","PN","OH","OO","DN_Price","Last_Sales","Last_GRR","Total_Calls","Total_Demands","FD_final",]
]

# ---------- normal branches ----------
data_branches = data[data["Brc"] != 20].copy()

data_branches = data_branches.merge(
    fd_branch_small,
    on=["Brc", "PN"],
    how="left"
)

data_branches["FD_final"] = data_branches["FD_final"].fillna(0)

data = pd.concat(
    [data_branches, national],
    ignore_index=True,
)

data = data.sort_values(
    ["Brc", "PN"],
    key=lambda s: s.astype(str),
).reset_index(drop=True)

branches = sorted(
    [b for b in data["Brc"].unique() if b != "National"]
) + ["National"]

## 3. OPM quadrant data

Each letter (A/B/C/D) is a quadrant: a Calls range plus its own 5-row Min Price / Max Price →
Service Level table — exactly the structure of your original input sheet. Edit these dicts
directly if the business rules change; the workbook is rebuilt from them every run. The PF table
never changes, so it's hardcoded straight into the Data-sheet formula instead of taking up space
on Parameters.

In [3]:
opm_quadrants = {
    "A": {"calls_min": 4, "calls_max": 6,
          "brackets": [(0, 11, 93), (11, 44, 90), (44, 119, 85), (119, 483, 83), (483, 999999, 80)]},
    "B": {"calls_min": 7, "calls_max": 11,
          "brackets": [(0, 11, 96), (11, 44, 95), (44, 119, 91), (119, 483, 87), (483, 999999, 81)]},
    "C": {"calls_min": 12, "calls_max": 26,
          "brackets": [(0, 11, 98), (11, 44, 96), (44, 119, 94), (119, 483, 92), (483, 999999, 87)]},
    "D": {"calls_min": 27, "calls_max": 999,
          "brackets": [(0, 11, 99), (11, 44, 97), (44, 119, 95), (119, 483, 93), (483, 999999, 89)]},
}

# Service Level -> PF (hardcoded — embedded directly into the Data-sheet PF formula, not on Parameters)
pf_lookup_values  = [80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99]
pf_lookup_results = [0.85,0.88,0.92,0.96,1,1.04,1.09,1.13,1.18,1.23,1.29,1.35,1.41,1.48,1.56,1.65,1.76,1.89,2.06,2.33]
PF_ARRAY_KEYS = "{" + ",".join(str(v) for v in pf_lookup_values) + "}"
PF_ARRAY_VALS = "{" + ",".join(str(v) for v in pf_lookup_results) + "}"

# Optionally carry forward previously entered LT / OCLT values
prior_lt = {}
prior_oclt = None
if PRIOR_PARAMETERS_FILE:
    _wb = load_workbook(PRIOR_PARAMETERS_FILE, data_only=True)
    _ws = _wb["Parameters"]
    _reading = False
    for row in _ws.iter_rows(values_only=True):
        if row[0] == "OCLT":
            prior_oclt = row[1]
        if row[0] == "Branch":
            _reading = True
            continue
        if _reading:
            if row[0] is None:
                break
            prior_lt[row[0]] = row[1]
    print(f"Carried forward LT for {len(prior_lt)} branches and OCLT={prior_oclt} from {PRIOR_PARAMETERS_FILE}")


## 4. Build the workbook

In [4]:
wb = Workbook()

FONT_NAME = "Arial"
HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10)
TITLE_FONT = Font(name=FONT_NAME, bold=True, size=12, color="1F4E78")
INPUT_FILL = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
BODY_FONT = Font(name=FONT_NAME, size=10)
BOLD_FONT = Font(name=FONT_NAME, size=10, bold=True)
THIN = Side(style="thin", color="B7B7B7")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
GREEN_BORDER = Border(left=Side(style="medium", color="375623"), right=Side(style="medium", color="375623"),
                       top=Side(style="medium", color="375623"), bottom=Side(style="medium", color="375623"))
CENTER = Alignment(horizontal="center", vertical="center")


def style_header_row(ws, row, col_start, col_end):
    for c in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = CENTER
        cell.border = BORDER


### 4a. Parameters sheet — quadrant grid

Renders the four OPM quadrants (A top-left, B top-right, C bottom-left, D bottom-right) in the
same visual layout as your original input sheet, including the vertical R-a-n-g-e label down the
left side of each block.

In [5]:
ws_p = wb.active
ws_p.title = "Parameters"
ws_p.sheet_view.showGridLines = False

# --- Title ---
ws_p.merge_cells("A1:O1")
title_cell = ws_p["A1"]
title_cell.value = "INPUT PARAMETER FOR OPM"
title_cell.font = Font(name=FONT_NAME, bold=True, size=13)
title_cell.alignment = CENTER

# --- Agency ---
ws_p["B3"] = "Agency"
ws_p["B3"].font = BOLD_FONT
#ws_p["D3"] = 23
ws_p["D3"].fill = INPUT_FILL
ws_p["D3"].border = BORDER
ws_p["D3"].alignment = CENTER


def build_quadrant(top_row, left_col, letter, calls_min, calls_max, brackets):
    """
    Renders one OPM quadrant (Calls row, headers, 5 price/SL rows) starting at
    (top_row, left_col), matching the layout of the original input sheet:
      col+0: vertical R/a/n/g/e/blank/<letter> label
      col+1: 'Calls' label / 'Min Price' header / min price values
      col+2: calls-min value / 'Max Price' header / max price values
      col+3: '-' / blank / blank
      col+4: calls-max value / blank / blank
      col+5: blank spacer
      col+6: 'Service Level' header / SL values
    Returns cell references used by the Data-sheet formulas.
    """
    c0, c1, c2, c3, c4, c5, c6 = [left_col + i for i in range(7)]

    r = top_row
    ws_p.cell(row=r, column=c0, value="R").font = BOLD_FONT
    ws_p.cell(row=r, column=c0).alignment = CENTER
    ws_p.cell(row=r, column=c1, value="Calls").font = BOLD_FONT
    calls_min_cell = ws_p.cell(row=r, column=c2, value=calls_min)
    calls_min_cell.fill = INPUT_FILL
    calls_min_cell.border = BORDER
    calls_min_cell.alignment = CENTER
    ws_p.cell(row=r, column=c3, value="-").alignment = CENTER
    calls_max_cell = ws_p.cell(row=r, column=c4, value=calls_max)
    calls_max_cell.fill = INPUT_FILL
    calls_max_cell.border = BORDER
    calls_max_cell.alignment = CENTER

    r += 1
    ws_p.cell(row=r, column=c0, value="a").font = BOLD_FONT
    ws_p.cell(row=r, column=c0).alignment = CENTER
    ws_p.cell(row=r, column=c1, value="Min Price").font = BOLD_FONT
    ws_p.cell(row=r, column=c2, value="Max Price").font = BOLD_FONT
    ws_p.cell(row=r, column=c3, value="Service Level").font = BOLD_FONT

    row_letters = ["n", "g", "e", "", letter]
    price_min_cells, price_max_cells, sl_cells = [], [], []
    for i, (pmin, pmax, sl) in enumerate(brackets):
        r += 1
        lbl = row_letters[i]
        lbl_cell = ws_p.cell(row=r, column=c0, value=lbl if lbl else None)
        lbl_cell.font = BOLD_FONT
        lbl_cell.alignment = CENTER
        pmin_cell = ws_p.cell(row=r, column=c1, value=pmin)
        pmin_cell.fill = INPUT_FILL
        pmin_cell.border = BORDER
        pmin_cell.alignment = CENTER
        pmax_cell = ws_p.cell(row=r, column=c2, value=pmax)
        pmax_cell.fill = INPUT_FILL
        pmax_cell.border = BORDER
        pmax_cell.alignment = CENTER
        sl_cell = ws_p.cell(row=r, column=c3, value=sl)
        sl_cell.fill = INPUT_FILL
        sl_cell.border = BORDER
        sl_cell.alignment = CENTER
        price_min_cells.append(pmin_cell.coordinate)
        price_max_cells.append(pmax_cell.coordinate)
        sl_cells.append(sl_cell.coordinate)

    bottom_row = r
    return {
        "calls_min": f"Parameters!${get_column_letter(c2)}${top_row}",
        "calls_max": f"Parameters!${get_column_letter(c4)}${top_row}",
        "price_min_range": f"Parameters!${get_column_letter(c1)}${top_row+2}:${get_column_letter(c1)}${bottom_row}",
        "price_max_range": f"Parameters!${get_column_letter(c2)}${top_row+2}:${get_column_letter(c2)}${bottom_row}",
        "sl_range": f"Parameters!${get_column_letter(c3)}${top_row+2}:${get_column_letter(c3)}${bottom_row}",
        "bottom_row": bottom_row,
    }


TOP_ROW_1 = 5     # first quadrant row block (A / B)
LEFT_COL_1 = 1    # quadrant A starting column
LEFT_COL_2 = 9    # quadrant B starting column (gap between pairs)
GAP_ROWS = 1      # blank rows between the top pair and bottom pair

quad_refs = {}
quad_refs["A"] = build_quadrant(TOP_ROW_1, LEFT_COL_1, "A", **opm_quadrants["A"])
quad_refs["B"] = build_quadrant(TOP_ROW_1, LEFT_COL_2, "B", **opm_quadrants["B"])

TOP_ROW_2 = quad_refs["A"]["bottom_row"] + 1 + GAP_ROWS
quad_refs["C"] = build_quadrant(TOP_ROW_2, LEFT_COL_1, "C", **opm_quadrants["C"])
quad_refs["D"] = build_quadrant(TOP_ROW_2, LEFT_COL_2, "D", **opm_quadrants["D"])

r = quad_refs["C"]["bottom_row"] + 2

# --- OCLT: single value for the whole agency ---
ws_p.cell(row=r, column=1, value="OCLT").font = TITLE_FONT
r += 1
ws_p.cell(row=r, column=1, value="OCLT").font = BOLD_FONT
oclt_cell_row = r
oclt_cell = ws_p.cell(row=r, column=2, value=prior_oclt)
oclt_cell.fill = INPUT_FILL
oclt_cell.border = BORDER
r += 2
OCLT_CELL = f"Parameters!$B${oclt_cell_row}"

# --- Branch LT table ---
ws_p.cell(row=r, column=1,
          value="BRANCH LEAD TIME").font = TITLE_FONT
r += 1
for i, h in enumerate(["Branch", "LT"]):
    ws_p.cell(row=r, column=1 + i, value=h)
style_header_row(ws_p, r, 1, 2)
branch_start = r + 1
r += 1
for b in branches:
    ws_p.cell(row=r, column=1, value=b).font = BODY_FONT
    ws_p.cell(row=r, column=1).border = BORDER
    lt_cell = ws_p.cell(row=r, column=2, value=prior_lt.get(b))
    lt_cell.fill = INPUT_FILL
    lt_cell.border = BORDER
    r += 1
branch_end = r - 1

ws_p.column_dimensions["A"].width = 4
for c in ["B", "C", "D", "E", "G", "I", "J", "K", "L", "N"]:
    ws_p.column_dimensions[c].width = 11
for c in ["F", "H", "M"]:
    ws_p.column_dimensions[c].width = 3
ws_p.freeze_panes = "A2"

LT_RANGE = f"Parameters!$A${branch_start}:$B${branch_end}"

print("Parameters sheet built.")


Parameters sheet built.


### 4b. Data sheet

One row per Brc + P/N. The first 10 columns are values written straight from the merged data.
The last 8 columns (`OPM` → `Max`) are **live Excel formulas** referencing the Parameters quadrant
grid. `ExDlt`, `OC`, `Min`, `ROP`, and `Max` are all rounded to whole numbers.

In [6]:
ws_d = wb.create_sheet("Data")
ws_d.sheet_view.showGridLines = False

data_headers = ["Brc", "Agc", "PN", "OH", "OO", "DN_Price", "Last_Sales", "Last_GRR",
                 "Total_Calls", "Total_Demands", "FD_final", "OPM", "SL", "PF", "ExDlt", "OC", "Min", "ROP", "Max"]
for i, h in enumerate(data_headers):
    ws_d.cell(row=1, column=1 + i, value=h)
style_header_row(ws_d, 1, 1, len(data_headers))
ws_d.freeze_panes = "A2"

col = {h: i + 1 for i, h in enumerate(data_headers)}
def L(colname):
    return get_column_letter(col[colname])

n_rows = len(data)
start_row = 2

# --- raw data columns ---
raw_cols = ["Brc", "Agc", "PN", "OH", "OO", "DN_Price", "Last_Sales", "Last_GRR", "Total_Calls", "Total_Demands", "FD_final"]
for cname in raw_cols:
    excel_col = col[cname]
    for i, v in enumerate(data[cname].tolist()):
        ws_d.cell(row=start_row + i, column=excel_col, value=v).font = BODY_FONT

# --- formula columns ---
for i in range(n_rows):
    row = start_row + i
    calls_cell = f"{L('Total_Calls')}{row}"
    price_cell = f"{L('DN_Price')}{row}"
    brc_cell = f"{L('Brc')}{row}"
    fd_cell = f"{L('FD_final')}{row}"
    opm_cell = f"{L('OPM')}{row}"
    sl_cell = f"{L('SL')}{row}"
    pf_cell = f"{L('PF')}{row}"
    exdlt_cell = f"{L('ExDlt')}{row}"
    oc_cell = f"{L('OC')}{row}"
    min_cell = f"{L('Min')}{row}"
    rop_cell = f"{L('ROP')}{row}"

    # OPM = which quadrant's Calls range the part's Total Calls falls into (A -> B -> C -> D)
    ws_d[opm_cell] = (
        f"=IF({calls_cell}<={quad_refs['A']['calls_max']},\"A\","
        f"IF({calls_cell}<={quad_refs['B']['calls_max']},\"B\","
        f"IF({calls_cell}<={quad_refs['C']['calls_max']},\"C\",\"D\")))"
    )

    # SL = price-bracket lookup within the matching quadrant — the quadrant IS the letter,
    # selected via nested IF, so no separate Range column is needed
    def quad_sl(letter):
        q = quad_refs[letter]
        return (f"SUMPRODUCT(({price_cell}>={q['price_min_range']})*"
                f"({price_cell}<{q['price_max_range']})*{q['sl_range']})")

    ws_d[sl_cell] = (
        f"=IFERROR(IF({opm_cell}=\"A\",{quad_sl('A')},"
        f"IF({opm_cell}=\"B\",{quad_sl('B')},"
        f"IF({opm_cell}=\"C\",{quad_sl('C')},{quad_sl('D')}))),0)"
    )

    # PF: hardcoded table (never changes), embedded directly in the formula
    ws_d[pf_cell] = f"=IFERROR(LOOKUP({sl_cell},{PF_ARRAY_KEYS},{PF_ARRAY_VALS}),0)"

    # ExDlt / OC / Min / ROP / Max — all rounded to whole numbers
    ws_d[exdlt_cell] = f"=IFERROR(ROUND({fd_cell}*VLOOKUP({brc_cell},{LT_RANGE},2,FALSE)/30,0),0)"
    ws_d[oc_cell] = f"=IFERROR(ROUND({OCLT_CELL}*{fd_cell}/14,0),0)"
    ws_d[min_cell] = f"=ROUND({pf_cell}*SQRT({exdlt_cell})+{oc_cell},0)"
    ws_d[rop_cell] = f"=ROUND({min_cell}+{exdlt_cell},0)"
    ws_d[f"{L('Max')}{row}"] = f"=ROUND({min_cell}+{rop_cell},0)"

    for cname in ["OPM", "SL", "PF", "ExDlt", "OC", "Min", "ROP", "Max"]:
        ws_d.cell(row=row, column=col[cname]).font = BODY_FONT

# --- widths & number formats ---
widths = {"Brc": 8, "Agc": 8, "PN": 16, "OH": 8, "OO": 8, "DN_Price": 12, "Last_Sales": 13,
          "Last_GRR": 13, "Total_Calls": 12, "FD_final": 10, "OPM": 8, "SL": 8, "PF": 8,
          "ExDlt": 10, "OC": 10, "Min": 10, "ROP": 10, "Max": 10}
for cname, w in widths.items():
    ws_d.column_dimensions[L(cname)].width = w

for cname in ["Last_Sales", "Last_GRR"]:
    for i in range(n_rows):
        ws_d.cell(row=start_row + i, column=col[cname]).number_format = "mm/dd/yyyy"
for i in range(n_rows):
    ws_d.cell(row=start_row + i, column=col["DN_Price"]).number_format = "#,##0.00"
for cname in ["ExDlt", "OC", "Min", "ROP", "Max"]:
    for i in range(n_rows):
        ws_d.cell(row=start_row + i, column=col[cname]).number_format = "#,##0"

DATA_LAST_ROW = start_row + n_rows - 1
D = {h: f"Data!${L(h)}$2:${L(h)}${DATA_LAST_ROW}" for h in data_headers}

print(f"Data sheet built ({n_rows:,} rows).")


Data sheet built (31,605 rows).


### 4c. Dashboard sheet

A single-part lookup card: enter **Agency / Branch / PN**, and the output row (OH, OO, DN Price,
Total Calls, Total Demands, FD Final, Min, Max, ExDlt, ROP, Last Sales, Last GRR, **SOQ**, **Condition**) pulls live from the Data sheet.

**Condition** is a single mutually-exclusive status per part — only one of Overstock / Deadstock /
Critical / Reorder can ever be true, in that priority order: Deadstock always wins outright;
otherwise Overstock is checked, then Critical, then Reorder (so Critical suppresses Reorder). The
four checkboxes below the card simply reflect this single Condition value, so exactly one (or none)
is ever checked.

**SOQ** (Suggested Order Quantity) depends on Condition:
- Overstock → `OH - Max` (negative = excess to pull out)
- Deadstock → `"-"`
- Critical → `ROP - OH - OO` tagged `(AF)` for Air Freight
- Reorder → `Max - OH - OO` tagged `(SF)` for Sea Freight

Below the card, an **All Branches** table lists every branch in the selected Agency for the
selected PN (Branch, PN, OH, OO, Total Calls, Total Demand, Min, Max, FD Final, ExDlt, ROP,
Last Sales, Last GRR, Condition), using the same mutually-exclusive Condition logic per row —
handy for spotting interbranch transfer opportunities (deadstock/overstock at one branch vs.
critical/reorder at another) at a glance.


In [ ]:
ws_b = wb.create_sheet("Dashboard")
ws_b.sheet_view.showGridLines = False

# --- selectors: Agency / Branch / PN ---
ws_b["A1"] = "Agency"
ws_b["A1"].font = Font(name=FONT_NAME, bold=True)
ws_b["B1"] = 23
ws_b["B1"].fill = INPUT_FILL
ws_b["B1"].border = GREEN_BORDER
ws_b["B1"].font = BODY_FONT

ws_b["D1"] = "Branch"
ws_b["D1"].font = Font(name=FONT_NAME, bold=True)
ws_b["E1"].fill = INPUT_FILL
ws_b["E1"].border = GREEN_BORDER
ws_b["E1"].font = BODY_FONT

ws_b["G1"] = "PN"
ws_b["G1"].font = Font(name=FONT_NAME, bold=True)
ws_b["H1"].fill = INPUT_FILL
ws_b["H1"].border = GREEN_BORDER
ws_b["H1"].font = BODY_FONT

# Branch dropdown, sourced from the Parameters branch table
dv_branch = DataValidation(type="list", formula1=f"=Parameters!$A${branch_start}:$A${branch_end}", allow_blank=True)
ws_b.add_data_validation(dv_branch)
dv_branch.add(ws_b["E1"])

AGENCY = "$B$1"
BRANCH = "$E$1"
PN = "$H$1"

# --- output row: single-part lookup card ---
out_fields = [
    ("OH", "OH"),
    ("OO", "OO"),
    ("DN Price", "DN_Price"),
    ("Total Calls", "Total_Calls"),
    ("Total Demands", "Total_Demands"),
    ("FD Final", "FD_final"),
    ("Min", "Min"),
    ("Max", "Max"),
    ("ExDlt", "ExDlt"),
    ("ROP", "ROP"),
    ("Last Sales", "Last_Sales"),
    ("Last GRR", "Last_GRR"),
]
header_row = 3
value_row = 4
for i, (label, _) in enumerate(out_fields):
    cell = ws_b.cell(row=header_row, column=1 + i, value=label)
    cell.font = Font(name=FONT_NAME, bold=True)
    cell.alignment = Alignment(horizontal="center")

# Shared row-match position for every lookup in the output row (single SUMPRODUCT match)
match_formula = (
    f"SUMPRODUCT(({D['Agc']}={AGENCY})*({D['Brc']}={BRANCH})*({D['PN']}={PN})*ROW({D['PN']}))"
    f"-{start_row}+1"
)
for i, (label, fieldname) in enumerate(out_fields):
    cell = ws_b.cell(row=value_row, column=1 + i)
    cell.value = f"=IFERROR(INDEX({D[fieldname]},{match_formula}),\"\")"
    cell.fill = INPUT_FILL
    cell.border = BORDER
    cell.font = BODY_FONT
    cell.alignment = Alignment(horizontal="center")
    if fieldname in ("Last_Sales", "Last_GRR"):
        cell.number_format = "mm/dd/yyyy"
    elif fieldname == "DN_Price":
        cell.number_format = "#,##0.00"
    elif fieldname in ("ExDlt", "OC", "Min", "ROP", "Max"):
        cell.number_format = "#,##0"

# Cell references to the looked-up output-row values, by field name (reused everywhere below)
OUT = {fieldname: f"${get_column_letter(1+i)}${value_row}" for i, (_, fieldname) in enumerate(out_fields)}

# --- Condition + SOQ columns, right of Last GRR ---
# Column layout: ... L=Last GRR, M=SOQ, N=Condition
soq_col = len(out_fields) + 1        # M
cond_col = soq_col + 1               # N

soq_header_cell = ws_b.cell(row=header_row, column=soq_col, value="SOQ")
soq_header_cell.font = Font(name=FONT_NAME, bold=True)
soq_header_cell.alignment = Alignment(horizontal="center")

cond_header_cell = ws_b.cell(row=header_row, column=cond_col, value="Condition")
cond_header_cell.font = Font(name=FONT_NAME, bold=True)
cond_header_cell.alignment = Alignment(horizontal="center")

COND_CELL = f"${get_column_letter(cond_col)}${value_row}"

# Deadstock threshold: National = 36 months, other branches = 30 months.
# Uses the latest activity date = MAX(Last Sales, Last GRR)
deadstock_months = f'IF({BRANCH}="National",36,30)'
latest_activity = f"MAX({OUT['Last_Sales']},{OUT['Last_GRR']})"
deadstock_formula = (
    f"AND({latest_activity}<>0,{latest_activity}<EDATE(TODAY(),-{deadstock_months}))"
)

# Single priority-ranked condition for the selected part — a part can only ever be
# ONE of the four: Deadstock always wins outright; otherwise Overstock, then
# Critical, then Reorder are checked in order, so Critical suppresses Reorder.
condition_formula = (
    f'IF({deadstock_formula},"Deadstock",'
    f'IF(AND({OUT["OH"]}<>"",{OUT["OH"]}>{OUT["Max"]}),"Overstock",'
    f'IF(AND({OUT["OH"]}<>"",{OUT["OH"]}<{OUT["ExDlt"]}),"Critical",'
    f'IF(AND({OUT["OH"]}<>"",{OUT["OH"]}<{OUT["ROP"]}),"Reorder",""))))'
)
cond_value_cell = ws_b.cell(row=value_row, column=cond_col, value=f"={condition_formula}")
cond_value_cell.fill = INPUT_FILL
cond_value_cell.border = BORDER
cond_value_cell.font = BODY_FONT
cond_value_cell.alignment = Alignment(horizontal="center")

# SOQ (Suggested Order Quantity):
#   Overstock -> OH - Max               (negative = excess to pull out)
#   Deadstock -> "-"                    (no order suggested)
#   Critical  -> ROP - OH - OO  (AF)    (Air Freight - urgent replenishment)
#   Reorder   -> Max - OH - OO  (SF)    (Sea Freight - normal replenishment)
soq_formula = (
    f'IF({COND_CELL}="Overstock",{OUT["OH"]}-{OUT["Max"]},'
    f'IF({COND_CELL}="Deadstock","-",'
    f'IF({COND_CELL}="Critical",TEXT({OUT["ROP"]}-{OUT["OH"]}-{OUT["OO"]},"#,##0")&" (AF)",'
    f'IF({COND_CELL}="Reorder",TEXT({OUT["Max"]}-{OUT["OH"]}-{OUT["OO"]},"#,##0")&" (SF)",""))))'
)
soq_value_cell = ws_b.cell(row=value_row, column=soq_col, value=f"={soq_formula}")
soq_value_cell.fill = INPUT_FILL
soq_value_cell.border = BORDER
soq_value_cell.font = BODY_FONT
soq_value_cell.alignment = Alignment(horizontal="center")

# --- Item Condition checkboxes (mutually exclusive — driven by COND_CELL) ---
cond_row0 = 7
ws_b.cell(row=cond_row0, column=1, value="Item Condition:").font = Font(name=FONT_NAME, bold=True, size=11)

labels = ["Overstock", "Deadstock", "Critical", "Reorder"]
for j, label in enumerate(labels):
    row = cond_row0 + 1 + j
    ws_b.cell(row=row, column=2, value=label).font = BODY_FONT
    chk_cell = ws_b.cell(row=row, column=3)
    chk_cell.value = f'=IF({COND_CELL}="{label}","\u2611","\u2610")'
    chk_cell.font = Font(name="Segoe UI Symbol", size=13, bold=True)
    chk_cell.alignment = Alignment(horizontal="center", vertical="center")
    chk_cell.border = GREEN_BORDER if j == 0 else BORDER

# =================================================================
# All-branches table: every branch in the selected Agency, for the selected PN,
# with its own condition (text only). Mirrors the Brc/PN/OH/OO/... layout below
# the checkboxes, matching the requested mock-up.
# =================================================================
table_title_row = cond_row0 + len(labels) + 2   # 2-row gap under the checkboxes
ws_b.cell(row=table_title_row, column=1, value="All Branches").font = Font(
    name=FONT_NAME, bold=True, size=12, color="1F4E78"
)

table_header_row = table_title_row + 1
table_data_start = table_header_row + 1

table_columns = [
    ("Branch", "Brc"),
    ("PN", None),
    ("OH", "OH"),
    ("OO", "OO"),
    ("Total Calls", "Total_Calls"),
    ("Total Demand", "Total_Demands"),
    ("Min", "Min"),
    ("Max", "Max"),
    ("FD Final", "FD_final"),
    ("ExDlt", "ExDlt"),
    ("ROP", "ROP"),
    ("Last Sales", "Last_Sales"),
    ("Last GRR", "Last_GRR"),
    ("Condition", None),
]
for i, (label, _) in enumerate(table_columns):
    ws_b.cell(row=table_header_row, column=1 + i, value=label)
style_header_row(ws_b, table_header_row, 1, len(table_columns))
ws_b.freeze_panes = ws_b.cell(row=table_data_start, column=1).coordinate

COL_BRC, COL_PN, COL_OH, COL_OO, COL_CALLS, COL_DEM, COL_MIN, COL_MAX, COL_FD, \
    COL_EXDLT, COL_ROP, COL_LS, COL_LG, COL_COND = range(1, 1 + len(table_columns))

lookup_fields = {
    COL_OH: "OH", COL_OO: "OO", COL_CALLS: "Total_Calls", COL_DEM: "Total_Demands",
    COL_MIN: "Min", COL_MAX: "Max", COL_FD: "FD_final", COL_EXDLT: "ExDlt",
    COL_ROP: "ROP", COL_LS: "Last_Sales", COL_LG: "Last_GRR",
}

for idx, b in enumerate(branches):
    row = table_data_start + idx

    brc_cell = ws_b.cell(row=row, column=COL_BRC, value=b)
    brc_cell.font = BODY_FONT
    brc_cell.border = BORDER
    brc_cell.alignment = Alignment(horizontal="center")
    brc_ref = f"${get_column_letter(COL_BRC)}${row}"

    pn_cell = ws_b.cell(row=row, column=COL_PN, value=f"={PN}")
    pn_cell.font = BODY_FONT
    pn_cell.border = BORDER
    pn_cell.alignment = Alignment(horizontal="center")

    row_match = (
        f"SUMPRODUCT(({D['Agc']}={AGENCY})*({D['Brc']}={brc_ref})*({D['PN']}={PN})*ROW({D['PN']}))"
        f"-{start_row}+1"
    )
    for col_i, fieldname in lookup_fields.items():
        cell = ws_b.cell(row=row, column=col_i)
        cell.value = f"=IFERROR(INDEX({D[fieldname]},{row_match}),\"\")"
        cell.border = BORDER
        cell.font = BODY_FONT
        cell.alignment = Alignment(horizontal="center")
        if fieldname in ("Last_Sales", "Last_GRR"):
            cell.number_format = "mm/dd/yyyy"
        elif fieldname in ("Min", "Max", "ExDlt", "ROP"):
            cell.number_format = "#,##0"

    oh_ref = f"${get_column_letter(COL_OH)}${row}"
    max_ref = f"${get_column_letter(COL_MAX)}${row}"
    exdlt_ref = f"${get_column_letter(COL_EXDLT)}${row}"
    rop_ref = f"${get_column_letter(COL_ROP)}${row}"
    ls_ref = f"${get_column_letter(COL_LS)}${row}"
    lg_ref = f"${get_column_letter(COL_LG)}${row}"

    deadstock_months_row = f'IF({brc_ref}="National",36,30)'
    latest_activity_row = f"MAX({ls_ref},{lg_ref})"
    deadstock_row_formula = (
        f"AND({latest_activity_row}<>0,{latest_activity_row}<EDATE(TODAY(),-{deadstock_months_row}))"
    )

    # Same mutually-exclusive priority as the single-part card: Deadstock > Overstock > Critical > Reorder
    condition_row_formula = (
        f'IFERROR(IF({deadstock_row_formula},"Deadstock",'
        f'IF(AND({oh_ref}<>"",{oh_ref}>{max_ref}),"Overstock",'
        f'IF(AND({oh_ref}<>"",{oh_ref}<{exdlt_ref}),"Critical",'
        f'IF(AND({oh_ref}<>"",{oh_ref}<{rop_ref}),"Reorder","")))),"")'
    )
    cond_cell = ws_b.cell(row=row, column=COL_COND, value=f"={condition_row_formula}")
    cond_cell.border = BORDER
    cond_cell.font = BODY_FONT
    cond_cell.alignment = Alignment(horizontal="center")

table_data_end = table_data_start + len(branches) - 1

# --- column widths (shared by the top card and the branch table below it) ---
widths = {1: 10, 2: 16, 3: 12, 4: 12, 5: 14, 6: 13, 7: 10, 8: 10,
          9: 10, 10: 10, 11: 13, 12: 13, 13: 14, 14: 14}
for c, w in widths.items():
    ws_b.column_dimensions[get_column_letter(c)].width = w

wb.save(OUTPUT_FILE)
print(f"Saved {OUTPUT_FILE}  ({n_rows:,} data rows, {len(branches)} branches in Dashboard table)")


## 5. Recalculate & verify

openpyxl writes formulas but not their cached results, so we run LibreOffice headless to
recalculate the file in place and confirm there are zero formula errors before handing it off.

In [8]:
import subprocess

result = subprocess.run(
    ["python3", "/mnt/skills/public/xlsx/scripts/recalc.py", OUTPUT_FILE, "180"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)



Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.



## 6. Next steps

1. Open **`Parameters`** and fill in the single **OCLT** cell and **LT** per branch (yellow cells).
   The A/B/C/D quadrant tables at the top mirror your original input sheet — edit the Calls range,
   Min/Max Price brackets, or Service Levels there directly if the business rules ever change.
2. Use the **`Dashboard`** sheet to look up any Branch + PN — pick a branch from the dropdown,
   type a PN, and the output row plus the four checkboxes update live.
3. The **`Data`** sheet's OPM / SL / PF / ExDlt / OC / Min / ROP / Max columns recalculate live in
   Excel as soon as you change LT, OCLT, or the OPM quadrant tables.
4. Re-run this notebook whenever you have a fresh `pmovdcE` / `FD` export. Set
   `PRIOR_PARAMETERS_FILE` in cell 2 to your previously filled-in workbook to carry LT/OCLT forward
   automatically.